## Industry rule for this stage

**Rule:** any binning, ratio, or derived-column logic must be defined once and applied identically to both train and test — the function that creates a feature should never see the word "test" branch differently except for which frozen train-fit parameters it re-uses.


In [ ]:
# import required libraries and add project root to the path
import sys
sys.path.append("../")

from multiple_stage_notebooks import *

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
# load the cleaned train/test datasets and preview the train set
capped_train_df_copy = pd.read_csv("../data/processed/capped_train_df.csv")
capped_test_df_copy = pd.read_csv("../data/processed/capped_test_df.csv")

no_capped_train_df_copy = pd.read_csv("../data/processed/no_capped_train_df.csv")
no_capped_test_df_copy = pd.read_csv("../data/processed/no_capped_test_df.csv")

capped_train_df_copy.head()

,Age,Income,Home_ownership,Employment_length,Loan_intent,internal_credit_rating,Loan_amount,Interest_rate,Loan_percent_income,Default,Credit_history_length,Loan_status
0,23.0,75800.0,Rent,0.0,Personal,A,7000.0,6.54,0.09,N,2.0,False
1,25.0,61959.0,Rent,9.0,Education,C,25000.0,12.73,0.40,Y,4.0,True
2,31.0,53088.0,Rent,0.0,Personal,A,6000.0,6.54,0.11,N,9.0,False
3,27.0,225000.0,Mortgage,12.0,Homeimprovement,A,6000.0,7.14,0.03,N,6.0,False
4,29.0,54000.0,Rent,0.0,Personal,D,10000.0,14.96,0.19,N,9.0,True


# Check point 3 - Feature Engineering

In [5]:
# check available columns
capped_train_df_copy.columns

Index(['Age', 'Income', 'Home_ownership', 'Employment_length', 'Loan_intent',
       'internal_credit_rating', 'Loan_amount', 'Interest_rate',
       'Loan_percent_income', 'Default', 'Credit_history_length',
       'Loan_status'],
      dtype='object')

In [6]:
# check unique Age values sorted in descending order
capped_train_df_copy['Age'].sort_values(ascending=False).unique()

array([94., 84., 76., 73., 70., 69., 66., 65., 64., 63., 62., 61., 60.,
       59., 58., 57., 56., 55., 54., 53., 52., 51., 50., 49., 48., 47.,
       46., 45., 44., 43., 42., 41., 40., 39., 38., 37., 36., 35., 34.,
       33., 32., 31., 30., 29., 28., 27., 26., 25., 24., 23., 22., 21.,
       20.])

Standard age range is usually 18–60 years, extendable up to 65–70 depending on contex

In [7]:
# create eligibility flag based on age range (18-60)
capped_train_df_copy['is_eligible'] = (capped_train_df_copy['Age'] >= 18) & (capped_train_df_copy['Age'] <= 60)
capped_test_df_copy['is_eligible'] = (capped_test_df_copy['Age'] >= 18) & (capped_test_df_copy['Age'] <= 60)

no_capped_train_df_copy['is_eligible'] = (no_capped_train_df_copy['Age'] >= 18) & (no_capped_train_df_copy['Age'] <= 60)
no_capped_test_df_copy['is_eligible'] = (no_capped_test_df_copy['Age'] >= 18) & (no_capped_test_df_copy['Age'] <= 60)

In [8]:
# Eligible persons to get the loan (Non Default Person)
capped_train_df_copy[(capped_train_df_copy['Loan_status'] == False) & (capped_train_df_copy['is_eligible'] == True)]

,Age,Income,Home_ownership,Employment_length,Loan_intent,internal_credit_rating,Loan_amount,Interest_rate,Loan_percent_income,Default,Credit_history_length,Loan_status,is_eligible
0,23.0,75800.0,Rent,0.0,Personal,A,7000.0,6.54,0.09,N,2.0,False,True
2,31.0,53088.0,Rent,0.0,Personal,A,6000.0,6.54,0.11,N,9.0,False,True
3,27.0,225000.0,Mortgage,12.0,Homeimprovement,A,6000.0,7.14,0.03,N,6.0,False,True
5,25.0,90000.0,Mortgage,4.0,Personal,B,3000.0,11.49,0.03,N,4.0,False,True
6,21.0,54036.0,Mortgage,3.0,Medical,B,8000.0,11.36,0.15,N,2.0,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25927,23.0,25000.0,Mortgage,4.0,Medical,A,5000.0,9.32,0.20,N,2.0,False,True
25928,22.0,21600.0,Mortgage,4.0,Education,A,4025.0,5.42,0.19,N,4.0,False,True
25929,28.0,81000.0,Rent,13.0,Venture,A,6225.0,7.90,0.08,N,7.0,False,True
25930,30.0,100000.0,Mortgage,13.0,Personal,A,10000.0,6.17,0.10,N,9.0,False,True


In [9]:
# On Train
capped_train_df_copy['age_employment_ratio']   = capped_train_df_copy['Employment_length'] / capped_train_df_copy['Age']
capped_train_df_copy['income_per_credit_year'] = capped_train_df_copy['Income'] / (capped_train_df_copy['Credit_history_length'])

# On Test - same formulas
capped_test_df_copy['age_employment_ratio']    = capped_test_df_copy['Employment_length'] / capped_test_df_copy['Age']
capped_test_df_copy['income_per_credit_year']  = capped_test_df_copy['Income'] / (capped_test_df_copy['Credit_history_length'])

In [10]:
# On Train
no_capped_train_df_copy['age_employment_ratio']   = no_capped_train_df_copy['Employment_length'] / no_capped_train_df_copy['Age']
no_capped_train_df_copy['income_per_credit_year'] = no_capped_train_df_copy['Income'] / (no_capped_train_df_copy['Credit_history_length'])

# On Test - same formulas
no_capped_test_df_copy['age_employment_ratio']    = no_capped_test_df_copy['Employment_length'] / no_capped_test_df_copy['Age']
no_capped_test_df_copy['income_per_credit_year']  = no_capped_test_df_copy['Income'] / (no_capped_test_df_copy['Credit_history_length'])

In [11]:
# check skewness of continuous features
capped_train_df_copy[capped_train_df_copy.select_dtypes(include=['float', 'int']).columns].skew(axis=0, skipna=True)

Age                       1.926924
Income                    1.718862
Employment_length         1.254721
Loan_amount               1.005903
Interest_rate             0.205284
Loan_percent_income       1.053543
Credit_history_length     1.664383
age_employment_ratio      0.591115
income_per_credit_year    2.106697
dtype: float64

Observation - Interest rate is normal,

Income
Employment_length
Loan_amount
Loan_percent_income
Credit_history_length
final_credit_score
income_per_credit_year

all are highly skewed

In [12]:
# Save dataframe
capped_train_df_copy.to_csv("../data/processed/engineered_clean_train_df.csv", index=False)
capped_test_df_copy.to_csv("../data/processed/engineered_clean_test_df.csv", index=False)

# Save dataframe
no_capped_train_df_copy.to_csv("../data/processed/engineered_raw_train_df.csv", index=False)
no_capped_test_df_copy.to_csv("../data/processed/engineered_raw_test_df.csv", index=False)


In [13]:
# check available columns after feature engineering
capped_test_df_copy.columns

Index(['Age', 'Income', 'Home_ownership', 'Employment_length', 'Loan_intent',
       'internal_credit_rating', 'Loan_amount', 'Interest_rate',
       'Loan_percent_income', 'Default', 'Credit_history_length',
       'Loan_status', 'is_eligible', 'age_employment_ratio',
       'income_per_credit_year'],
      dtype='object')

In [14]:
capped_train_df_copy.columns

Index(['Age', 'Income', 'Home_ownership', 'Employment_length', 'Loan_intent',
       'internal_credit_rating', 'Loan_amount', 'Interest_rate',
       'Loan_percent_income', 'Default', 'Credit_history_length',
       'Loan_status', 'is_eligible', 'age_employment_ratio',
       'income_per_credit_year'],
      dtype='object')